<a href="https://colab.research.google.com/github/vinodrajs001/agentcore001/blob/main/event_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# https://youtu.be/bu2cD1pCFTs

In [2]:
%%writefile requirements.txt
boto3
pandas
requests
ipython
botocore
strands-agents
bedrock-agentcore
bedrock-agentcore-starter-toolkit

Writing requirements.txt


In [3]:
%pip install -qUr requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 96.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 627.8/627.8 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 100.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 568.4/568.4 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 429.6/429.6 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 501.6/501.6 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 131.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 124.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5

In [4]:
# Import the userdata module
from google.colab import userdata
import os

# Retrieve AWS credentials from Colab Secrets
aws_access_key_id = userdata.get('AWS_ACCESS_KEY_ID')
aws_secret_access_key = userdata.get('AWS_SECRET_ACCESS_KEY')
aws_region_name = userdata.get('AWS_REGION_NAME')


# Set the AWS environment variables
os.environ['AWS_ACCESS_KEY_ID'] = aws_access_key_id
os.environ['AWS_SECRET_ACCESS_KEY'] = aws_secret_access_key
os.environ['AWS_REGION_NAME'] = aws_region_name

print("AWS credentials loaded and configured from Colab Secrets.")


AWS credentials loaded and configured from Colab Secrets.


In [5]:
import os
import json
import uuid
import time
import boto3
import logging
import pandas as pd
import requests
from io import StringIO
from datetime import datetime
# from utils import create_agentcore_role
from IPython.display import Markdown,display

from botocore.exceptions import ClientError


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

logger = logging.getLogger("awssome-event-agent")


REGION = os.getenv('AWS_REGION', 'us-east-1')
UNIQUE_ID = str(uuid.uuid4())[:8]
# MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
MODEL_ID = "us.amazon.nova-micro-v1:0"
EMBEDDING_MODEL = "amazon.titan-embed-text-v2:0"


print(f"REGION: {REGION}")
print(f"UNIQUE_ID: {UNIQUE_ID}")
print(f"MODEL_ID: {MODEL_ID}")
print(f"EMBEDDING_MODEL: {EMBEDDING_MODEL}")


REGION: us-east-1
UNIQUE_ID: 002de403
MODEL_ID: us.amazon.nova-micro-v1:0
EMBEDDING_MODEL: amazon.titan-embed-text-v2:0


In [6]:
from strands import Agent
from strands.models import BedrockModel

model = BedrockModel(model_id=MODEL_ID)

simple_agent = Agent(
    model=model,
    system_prompt="You are an intelligent event assistant"
)

print("\n Simple Agent Created")


print("\n Testing simle agent \n")
response = simple_agent("which sessions is good to learbout about security with AI Agents, provide answer in 2 -3 lines")



 Simple Agent Created

 Testing simle agent 

Look for sessions titled "AI Ethics and Security" or "AI Agent Risk Management." These sessions typically cover the latest developments in safeguarding AI systems and addressing potential vulnerabilities and ethical concerns associated with AI agents.

In [22]:
# Create Knowledge Base
bedrock_agent_client = boto3.client(service_name='bedrock-agent', region_name=REGION)
KB_NAME = f"EmployeeDetailsKB_{UNIQUE_ID}"

kb_response = bedrock_agent_client.create_knowledge_base(
    name=KB_NAME,
    description="Knowledge base for employee details",
    roleArn="arn:aws:iam::610755329309:role/BedrockKBRole",
    knowledgeBaseConfiguration={
        'type': 'VECTOR',
        'vectorKnowledgeBaseConfiguration':{
            'embeddingModelArn': 'arn:aws:bedrock:us-east-1::foundation-model/amazon.titan-embed-text-v2:0',
            'embeddingModelConfiguration':{
                'bedrockEmbeddingModelConfiguration':{
                    'dimensions': 256,
                    'embeddingDataType': 'FLOAT32'
                }
            }
        }
    },
    storageConfiguration={
        'type': 'S3_VECTORS',
        's3VectorsConfiguration':{
            'indexArn': 'arn:aws:s3vectors:us-east-1:610755329309:bucket/bedrock-kb-001/index/employee-details-index'
        }
    }
)

KB_ID = kb_response['knowledgeBase']['knowledgeBaseId']
print(f"KB_ID: {KB_ID}")

KB_ID: 42IR4YKKL6


In [ ]:
kb = bedrock_agent_client.get_knowledge_base(
    knowledgeBaseId=KB_ID
)

print(kb["knowledgeBase"]["status"])

ds = bedrock_agent_client.get_data_source(
    knowledgeBaseId=KB_ID,
    dataSourceId=data_source_id
)

print(ds["dataSource"]["status"])

In [ ]:
ds_response= bedrock_agent_client.create_data_source(
    knowledgeBaseId=KB_ID,
    name=f"EmployeeDetailsDataSource_{UNIQUE_ID}",
    description="Data source for employee details",
    dataSourceConfiguration={
        'type': 'S3',
        's3Configuration': {
            'bucketArn': 'arn:aws:s3:::bedrock-agentcore-kb-610755329309-us-east-1-an'}},
    vectorIngestionConfiguration={
        'chunkingConfiguration':{
            'chunkingStrategy':'FIXED_SIZE',
            'fixedSizeChunkingConfiguration':{
                'maxTokens': 100,
                'overlapPercentage':10
            }
        }
    }
)

data_source_id=ds_response['dataSource']['dataSourceId']
print(f"Data source created: {data_source_id}")


ingestion_response = bedrock_agent_client.start_ingestion_job(
    knowledgeBaseId=KB_ID,
    dataSourceId=data_source_id
)


injection_job_id = ingestion_response['ingestionJob']['ingestionJobId']
print(f"Ingestion job started: {injection_job_id}")

while True:
    status = bedrock_agent_client.get_ingestion_job(
        knowledgeBaseId=KB_ID,
        dataSourceId=data_source_id,
        ingestionJobId=injection_job_id
    )

    job_status = status["ingestionJob"]["status"]
    print(job_status)

    if job_status in ["COMPLETE", "FAILED"]:
        break

    time.sleep(10)




In [21]:
bedrock_agent_runtime = boto3.client('bedrock-agent-runtime',region_name=REGION)

test_query = "List all employees who is on leave from 1st June 2026"

print(f"Test Query: {test_query}")

response = bedrock_agent_runtime.retrieve(
    knowledgeBaseId=KB_ID,
    retrievalQuery={'text': test_query},
    retrievalConfiguration={
        'vectorSearchConfiguration': {
            'numberOfResults': 3
        }
    }
)

print(f"Found {len(response['retrievalResults'])} results:\n")
for idx, result in enumerate[any](response['retrievalResults'],1):
    content = result['content']['text']
    score = result['score']
    print(f"{idx}. Score: {score:.3f}")
    print(f" {content[:200]}...\n")



Test Query: List all employees who is on leave from 1st June 2026


NameError: name 'KB_ID' is not defined

In [18]:
# KB search tool
from strands import tool
from typing import List, Dict, Any

@tool
def search_reinvent_sessions(query: str, max_results: int = 3) -> List[Dict[str, Any]]:
    """Search for employee details from knowledge based using semantic search.
    Use this tool when user ask for employee details.

    Args :
      query: The search query.
      max_results: The maximum number of results to return.

    Returns:
      A list of employee details.

    """

    try:
        logger.info(f"Searching KB with query :{query}")

        response = bedrock_agent_runtime.retrieve(
            knowledgeBaseId=KB_ID,
            retrievalQuery={'text': query},
            retrievalConfiguration={
                'vectorSearchConfiguration': {
                    'numberOfResults': min(max_results,10)
                }
            }
        )

        results = []
        for idx, item in enumerate[Any](response.get('retrievalResults',[]),1):
            result = {
                'rank': idx,
                'content': item.get('content',{}).get('text',''),
                'score': item.get('score',0.0)
            }
            results.append(result)
        logger.info(f"Found {len(results)} RIV sessions")
        return results
    except Exception as e:
        logger.error(f"Error searching KB: {e}")
        return [{"error": f"Failed to search KB : {str(e)}"}]
print("\n KB Search Tool Created")


 KB Search Tool Created


In [16]:
# Create agentcore memory

from bedrock_agentcore.memory.constants import StrategyType, ConversationalMessage , MessageRole
from bedrock_agentcore_starter_toolkit.operations.memory.manager import MemoryManager
from bedrock_agentcore.memory.session import MemorySessionManager

memory_manager = MemoryManager(region_name=REGION)

memory = memory_manager.get_or_create_memory(
    name=f"EventAgentMemory_{UNIQUE_ID}",
    strategies=[
        {
            StrategyType.USER_PREFERENCE.value: {
                "name": "UserPreferences",
                "namespaces": ["/users/{actorId}/preferences"],
                "description": "Captures customer preferences and behavior"
            }
        }
    ],
    description="Memory for Employee Agent with user preferences",
    event_expiry_days=30
)

MEMORY_ID = memory.get("id")

print(f"Memory ID: {MEMORY_ID}")



Created memory: EventAgentMemory_002de403-Sx0TR4Buho
INFO:bedrock_agentcore_starter_toolkit.operations.memory.manager:Created memory: EventAgentMemory_002de403-Sx0TR4Buho
Created memory EventAgentMemory_002de403-Sx0TR4Buho, waiting for ACTIVE status...
INFO:bedrock_agentcore_starter_toolkit.operations.memory.manager:Created memory EventAgentMemory_002de403-Sx0TR4Buho, waiting for ACTIVE status...
Waiting for memory EventAgentMemory_002de403-Sx0TR4Buho to return to ACTIVE state and strategies to reach terminal states...
INFO:bedrock_agentcore_starter_toolkit.operations.memory.manager:Waiting for memory EventAgentMemory_002de403-Sx0TR4Buho to return to ACTIVE state and strategies to reach terminal states...


[11:54:00]    ⏳ Memory: CREATING, Strategies: 0/1 active (10s elapsed)                             ]8;id=137905;file:///usr/local/lib/python3.12/dist-packages/bedrock_agentcore_starter_toolkit/operations/memory/manager.py\manager.py]8;;\:]8;id=483218;file:///usr/local/lib/python3.12/dist-packages/bedrock_agentcore_starter_toolkit/operations/memory/manager.py#1333\1333]8;;\

[11:54:10]    ⏳ Memory: CREATING, Strategies: 0/1 active (20s elapsed)                             ]8;id=407104;file:///usr/local/lib/python3.12/dist-packages/bedrock_agentcore_starter_toolkit/operations/memory/manager.py\manager.py]8;;\:]8;id=69898;file:///usr/local/lib/python3.12/dist-packages/bedrock_agentcore_starter_toolkit/operations/memory/manager.py#1333\1333]8;;\

[11:54:20]    ⏳ Memory: CREATING, Strategies: 0/1 active (30s elapsed)                             ]8;id=769390;file:///usr/local/lib/python3.12/dist-packages/bedrock_agentcore_starter_toolkit/operations/memory/manager.py\manager.py]8;;\:]8;id=941601;file:///usr/local/lib/python3.12/dist-packages/bedrock_agentcore_starter_toolkit/operations/memory/manager.py#1333\1333]8;;\

[11:54:30]    ⏳ Memory: CREATING, Strategies: 0/1 active (40s elapsed)                             ]8;id=691929;file:///usr/local/lib/python3.12/dist-packages/bedrock_agentcore_starter_toolkit/operations/memory/manager.py\manager.py]8;;\:]8;id=772311;file:///usr/local/lib/python3.12/dist-packages/bedrock_agentcore_starter_toolkit/operations/memory/manager.py#1333\1333]8;;\

...

Memory EventAgentMemory_002de403-Sx0TR4Buho is ACTIVE and all strategies are in terminal states (took 151 seconds)
INFO:bedrock_agentcore_starter_toolkit.operations.memory.manager:Memory EventAgentMemory_002de403-Sx0TR4Buho is ACTIVE and all strategies are in terminal states (took 151 seconds)


              ✅ Memory is ACTIVE (took 151s)                                                       ]8;id=108034;file:///usr/local/lib/python3.12/dist-packages/bedrock_agentcore_starter_toolkit/operations/memory/manager.py\manager.py]8;;\:]8;id=946075;file:///usr/local/lib/python3.12/dist-packages/bedrock_agentcore_starter_toolkit/operations/memory/manager.py#1353\1353]8;;\

ObservabilityDeliveryManager initialized for region: us-east-1, account: 610755329309
INFO:bedrock_agentcore_starter_toolkit.operations.observability.delivery:ObservabilityDeliveryManager initialized for region: us-east-1, account: 610755329309
Created log group: /aws/vendedlogs/bedrock-agentcore/memory/APPLICATION_LOGS/EventAgentMemory_002de403-Sx0TR4Buho
INFO:bedrock_agentcore_starter_toolkit.operations.observability.delivery:Created log group: /aws/vendedlogs/bedrock-agentcore/memory/APPLICATION_LOGS/EventAgentMemory_002de403-Sx0TR4Buho
Failed to enable observability for memory/EventAgentMemory_002de403-Sx0TR4Buho: AccessDeniedException - Access Denied for this Delivery Destination. Please make sure that you have correct permissions to access the Log Destination Resource.
ERROR:bedrock_agentcore_starter_toolkit.operations.observability.delivery:Failed to enable observability for memory/EventAgentMemory_002de403-Sx0TR4Buho: AccessDeniedException - Access Denied for this Delivery Dest

⚠️ Failed to enable observability: AccessDeniedException: Access Denied for this Delivery Destination. Please make 
sure that you have correct permissions to access the Log Destination Resource.

Memory ID: EventAgentMemory_002de403-Sx0TR4Buho


In [10]:
from botocore import context
from strands.hooks import (
    HookProvider,
    HookRegistry,
    AgentInitializedEvent,
    MessageAddedEvent
)

class MemoryHookProvider(HookProvider):
    # https://youtu.be/bu2cD1pCFTs?t=2198
    def __init__(self):
        logger.info("Initializing Memory Hook Provider")
        self.memory_session_manager = MemorySessionManager(MEMORY_ID,REGION)


    def on_agent_initialized(self, event: AgentInitializedEvent):
        logger.info("Agent Initialized")

        actor_id = event.agent.state.get("actor_id")

        if not actor_id:
            logger.warning("Missing actor_id")
            return
        try:
            preferences = self.memory_session_manager.search_long_term_memories(
                namespace_prefix=f"/users/{actor_id}/preferences",
                query="what are the user's preferences",
                top_k=5
            )

            if preferences:
                logger.info(f"User preferences: {preferences}")
                pref_messages = []
                for pref in preferences:
                    pref_text = pref.get('content',{}).get('text','')
                    if pref_text:
                        try:
                            pref_json = json.loads(pref_text)
                            pref_messages.append(f"- {pref_json.get('preferences', pref_text)}")
                        except:
                            pref_messages.append(f"{pref_text}\n")
                if pref_messages:
                    context = "\n".join(pref_messages)
                    event.agent.system_prompt += f"**User Preferences {context}"
                    logger.info("Addeded user preference")
            else:
                logger.info("No user preferences found")
        except Exception as e:
            logger.error(f"Error retrieving user preferences: {e}")

    def on_message_added(self, event: MessageAddedEvent):
        logger.info("Message Added")
        actor_id = event.agent.state.get("actor_id")
        session_id = event.agent.state.get("session_id")

        if not actor_id or not session_id:
            logger.warning("Missing actor_id or session_id")
            return

        try:
            messages = event.agent.messages
            last_message = messages[-1]
            message_content = str(last_message.get("content", ""))
            if last_message["role"] == "user":
                message_role = MessageRole.USER
            elif last_message["role"] == "assistant":
                message_role = MessageRole.ASSISTANT

            self.memory_session_manager.add_turns(
                actor_id=actor_id,
                session_id=session_id,
                messages=[
                    ConversationalMessage(
                      message_content, message_role
                    )
                ]
            )
            logger.info("Added message to memory")
        except Exception as e:
            logger.error(f"Error adding message to memory: {e}")

    def register_hooks(self, registry: HookRegistry):
        logger.info("Registering Memory Hooks")
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)
        registry.add_callback(MessageAddedEvent, self.on_message_added)

print("\n Memory Hook Provider Created")



 Memory Hook Provider Created


In [14]:
from botocore import context
from strands.hooks import (
    HookProvider,
    HookRegistry,
    AgentInitializedEvent,
    MessageAddedEvent
)

class MemoryHookProvider(HookProvider):
    # https://youtu.be/bu2cD1pCFTs?t=2198
    def __init__(self):
        logger.info("Initializing Memory Hook Provider")
        self.memory_session_manager = MemorySessionManager(MEMORY_ID,REGION)


    def on_agent_initialized(self, event: AgentInitializedEvent):
        logger.info("Agent Initialized")

        actor_id = event.agent.state.get("actor_id")

        if not actor_id:
            logger.warning("Missing actor_id")
            return
        try:
            preferences = self.memory_session_manager.search_long_term_memories(
                namespace_prefix=f"/users/{actor_id}/preferences",
                query="what are the user's preferences",
                top_k=5
            )

            if preferences:
                logger.info(f"User preferences: {preferences}")
                pref_messages = []
                for pref in preferences:
                    pref_text = pref.get('content',{}).get('text','')
                    if pref_text:
                        try:
                            pref_json = json.loads(pref_text)
                            pref_messages.append(f"- {pref_json.get('preferences', pref_text)}")
                        except:
                            pref_messages.append(f"{pref_text}\n")
                if pref_messages:
                    context = "\n".join(pref_messages)
                    event.agent.system_prompt += f"**User Preferences {context}"
                    logger.info("Addeded user preference")
            else:
                logger.info("No user preferences found")
        except Exception as e:
            logger.error(f"Error retrieving user preferences: {e}")

    def on_message_added(self, event: MessageAddedEvent):
        logger.info("Message Added")
        actor_id = event.agent.state.get("actor_id")
        session_id = event.agent.state.get("session_id")

        if not actor_id or not session_id:
            logger.warning("Missing actor_id or session_id")
            return

        try:
            messages = event.agent.messages
            last_message = messages[-1]
            message_content = str(last_message.get("content", ""))
            if last_message["role"] == "user":
                message_role = MessageRole.USER
            elif last_message["role"] == "assistant":
                message_role = MessageRole.ASSISTANT

            self.memory_session_manager.add_turns(
                actor_id=actor_id,
                session_id=session_id,
                messages=[
                    ConversationalMessage(
                      message_content, message_role
                    )
                ]
            )
            logger.info("Added message to memory")
        except Exception as e:
            logger.error(f"Error adding message to memory: {e}")

    def register_hooks(self, registry: HookRegistry):
        logger.info("Registering Memory Hooks")
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)
        registry.add_callback(MessageAddedEvent, self.on_message_added)

print("\n Memory Hook Provider Created")



 Memory Hook Provider Created


In [ ]:
# create agent with memory and RAG

ACTOR_ID = f"user-{UNIQUE_ID}"
session_id_1 = f"session-1-{str(uuid.uuid4())[:8]}"

print(f"ACTOR_ID: {ACTOR_ID}")
print(f"session_id_1: {session_id_1}")
print(f"memorry id: {MEMORY_ID}")

memory_hook = MemoryHookProvider()

# Create agent with memory & RAG

memory_rag_agent = Agent(
    model=model,
    system_prompt="You are an intelligent employee details assistant, you can retrieve emloyee details",
    hooks=[memory_hook],
    tools=[search_reinvent_sessions],
    state={
        "actor_id": ACTOR_ID,
        "session_id": session_id_1
    }
)

print("Memory enabled RAG agent created")



In [ ]:
# share personal information
# https://youtu.be/bu2cD1pCFTs?t=2539
response = memory_rag_agent(
    """Hi! Let me tell you about myself
      - My name is Vinod Shalgar
      - Work as a HR assistant
      - I am mainly looking for employee details such as leave balances, who is on leave
    """
)

In [52]:
from bedrock_agentcore.memory.models import MemoryRecord
# validate long term memory
# https://youtu.be/bu2cD1pCFTs?t=2573

memory_session_manager = MemorySessionManager(MEMORY_ID,REGION)

try:
    response = memory_session_manager.list_long_term_memory_records(
        namespace_prefix=f"/users/{ACTOR_ID}/preferences",
        max_results=10
    )

    if response:
            print(f"\n Found{len(response)} preferences")
            for idx, pref in enumerate[MemoryRecord](response,1):
                content = pref.get('content',{}).get('text','')
                print(f"{idx}. {content}")
    else:
        print("No preferences found")
except Exception as e:
    print(f"Error retrieving preferences: {e}")
# https://youtu.be/bu2cD1pCFTs?t=2614

/tmp/ipykernel_921/1063493774.py:8: DeprecationWarning: The 'namespace_prefix' parameter is deprecated and will be removed in a future release. Use 'namespace' for exact-match retrieval (current pre-redesign behavior during the service grace period) or 'namespace_path' for hierarchical path-prefix retrieval.
  response = memory_session_manager.list_long_term_memory_records(



 Found1 preferences
1. {"context":"Vinod Shalgar works as an HR assistant/recruiter. Initially noted as looking for resources in Python, Cloud Architecture, and hands-on roles; primarily uses the system to look up employee details such as leave balances and who is currently on leave.","preference":"Works in HR (assistant/recruiter role); primarily interested in employee HR data (leave balances, who is on leave); also interested in resources related to Python, Cloud Architecture, and hands-on roles","categories":["career","technology","recruitment","cloud","programming","work","HR","employee management"]}


In [23]:
# https://youtu.be/bu2cD1pCFTs?t=2645
# Cross Session Memory

session_id_2 = f"session-2-{str(uuid.uuid4())[:8]}"
print(f"session_id_2: {session_id_2}")
ACTOR_ID = f"user-{UNIQUE_ID}"

new_memory_hook = MemoryHookProvider()
new_memory_rag_agent = Agent(
    model=model,
    hooks=[new_memory_hook],
    tools=[search_reinvent_sessions],
    system_prompt="You are an intelligent employee details assistant, you can retrieve emloyee details",
    state={
        "actor_id": ACTOR_ID, # same user
        "session_id": session_id_2 # different sessions
    }
)
#https://youtu.be/bu2cD1pCFTs?t=2659
response = new_memory_rag_agent("List of employees who are on leave")
display(Markdown(f"**Agent:**{response.message['content'][0]['text']}"))



session_id_2: session-2-9a77469d


/tmp/ipykernel_1030/4090405752.py:25: DeprecationWarning: The 'namespace_prefix' parameter is deprecated and will be removed in a future release. Use 'namespace' for exact-match retrieval (current pre-redesign behavior during the service grace period) or 'namespace_path' for hierarchical path-prefix retrieval.
  preferences = self.memory_session_manager.search_long_term_memories(


<thinking> The User has asked for a list of employees who are on leave. To retrieve this information, I need to use the available tool, which can perform a semantic search for employee details. I will use the tool to search for employees with the query "on leave". I will set the maximum number of results to a default value of 3.</thinking>

Tool #1: search_reinvent_sessions
<thinking> I have received the result from the tool. It provides a list of employees who are currently on leave along with their leave details. Here are the employees who are on leave:

1. Priya Patel
   - Leave Type: Sick Leave
   - Start Date: 2026-05-15
   - End Date: 2026-05-16
   - Status: Approved

2. Rahul Sharma
   - Leave Type: Annual Leave
   - Start Date: 2026-06-01
   - End Date: 2026-06-05
   - Status: Approved

The third result provides leave policy information, which is not directly relevant to the current request. I will now provide this information to the User.</thinking>

The employees who are curr

**Agent:**<thinking> I have received the result from the tool. It provides a list of employees who are currently on leave along with their leave details. Here are the employees who are on leave:

1. Priya Patel
   - Leave Type: Sick Leave
   - Start Date: 2026-05-15
   - End Date: 2026-05-16
   - Status: Approved

2. Rahul Sharma
   - Leave Type: Annual Leave
   - Start Date: 2026-06-01
   - End Date: 2026-06-05
   - Status: Approved

The third result provides leave policy information, which is not directly relevant to the current request. I will now provide this information to the User.</thinking>

The employees who are currently on leave are:

1. Priya Patel with a sick leave from 2026-05-15 to 2026-05-16.
2. Rahul Sharma with an annual leave from 2026-06-01 to 2026-06-05. 

If you need more detailed information or have any other requests, feel free to ask!

In [24]:
# Try with another sessions
session_id_3 = f"session-2-{str(uuid.uuid4())[:8]}"
print(f"session_id_3: {session_id_3}")
ACTOR_ID = f"user-{UNIQUE_ID}"

new_memory_hook = MemoryHookProvider()
new_memory_rag_agent = Agent(
    model=model,
    hooks=[new_memory_hook],
    tools=[search_reinvent_sessions],
    system_prompt="You are an intelligent employee details assistant, you can retrieve emloyee details",
    state={
        "actor_id": ACTOR_ID, # same user
        "session_id": session_id_3 # different sessions
    }
)
#https://youtu.be/bu2cD1pCFTs?t=2659
response = new_memory_rag_agent("Provide me company leave policies")
display(Markdown(f"**Agent:**{response.message['content'][0]['text']}"))


session_id_3: session-2-e94e2ab8


/tmp/ipykernel_1030/4090405752.py:25: DeprecationWarning: The 'namespace_prefix' parameter is deprecated and will be removed in a future release. Use 'namespace' for exact-match retrieval (current pre-redesign behavior during the service grace period) or 'namespace_path' for hierarchical path-prefix retrieval.
  preferences = self.memory_session_manager.search_long_term_memories(


<thinking> The User is asking for company leave policies, which is not directly related to employee details. However, if the leave policies are associated with specific employees or roles, I can use the search tool to retrieve related employee details. Let me first confirm if this information can be retrieved from the tool.</thinking>

Tool #1: search_reinvent_sessions
<thinking> The tool has returned three results related to leave policies. The first result provides a general overview of the company's leave policies. The second result seems to be about travel policies and is not relevant. The third result provides specific leave details for an employee, Rahul Sharma, which might be useful if the User is interested in individual leave records.</thinking>

Based on the information retrieved, here are the company's leave policies:

**Company Leave Policies:**
- Employees are entitled to:
  - 20 annual leave days
  - 10 sick leave days
  - 5 casual leave days per year
- Manager approval i

**Agent:**<thinking> The tool has returned three results related to leave policies. The first result provides a general overview of the company's leave policies. The second result seems to be about travel policies and is not relevant. The third result provides specific leave details for an employee, Rahul Sharma, which might be useful if the User is interested in individual leave records.</thinking>

Based on the information retrieved, here are the company's leave policies:

**Company Leave Policies:**
- Employees are entitled to:
  - 20 annual leave days
  - 10 sick leave days
  - 5 casual leave days per year
- Manager approval is required for annual leave.

If you need more specific information or details about another employee's leave policies, please let me know!

In [ ]:
# AgentCore Runtime
# https://youtu.be/bu2cD1pCFTs?t=2783
